In [5]:
import pandas as pd
import glob
import csv

# Grab all the metadata files in your folder
all_info_files = glob.glob("*_info.csv")
match_data = []

print(f"Found {len(all_info_files)} match files. Processing...")

# Loop through every single file and extract the data we need
for file in all_info_files:
    match_id = file.split('_')[0]
    
    # Set default values in case a match was washed out or data is missing
    match_info = {
        'Match_ID': match_id, 'City': 'Unknown', 'Date': 'Unknown', 
        'Venue': 'Unknown', 'Toss_Winner': 'Unknown', 
        'Toss_Decision': 'Unknown', 'Winner': 'No Result'
    }
    teams = []
    
    # Read the CSV safely
    with open(file, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) >= 3:
                key = row[1]
                val = row[2]
                
                # Map the Cricsheet keys to our dictionary
                if key == 'city': match_info['City'] = val
                elif key == 'date': match_info['Date'] = val
                elif key == 'venue': match_info['Venue'] = val
                elif key == 'toss_winner': match_info['Toss_Winner'] = val
                elif key == 'toss_decision': match_info['Toss_Decision'] = val
                elif key == 'winner': match_info['Winner'] = val
                elif key == 'team': teams.append(val)
    
    # Assign Team A and Team B
    match_info['Team_A'] = teams[0] if len(teams) > 0 else 'Unknown'
    match_info['Team_B'] = teams[1] if len(teams) > 1 else 'Unknown'
    
    match_data.append(match_info)

# Convert our extracted data into a clean Pandas DataFrame
matches_df = pd.DataFrame(match_data)

# Save it as a single CSV file so we never have to run this loop again!
matches_df.to_csv('matches.csv', index=False)

print(f"\nSuccess! Built a clean dataset with {len(matches_df)} matches.")
display(matches_df.head())

Found 1169 match files. Processing...

Success! Built a clean dataset with 1169 matches.


,Match_ID,City,Date,Venue,Toss_Winner,Toss_Decision,Winner,Team_A,Team_B
0,1082591,Hyderabad,2017/04/05,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,Sunrisers Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore
1,1082592,Pune,2017/04/06,Maharashtra Cricket Association Stadium,Rising Pune Supergiant,field,Rising Pune Supergiant,Rising Pune Supergiant,Mumbai Indians
2,1082593,Rajkot,2017/04/07,Saurashtra Cricket Association Stadium,Kolkata Knight Riders,field,Kolkata Knight Riders,Gujarat Lions,Kolkata Knight Riders
3,1082594,Indore,2017/04/08,Holkar Cricket Stadium,Kings XI Punjab,field,Kings XI Punjab,Kings XI Punjab,Rising Pune Supergiant
4,1082595,Bengaluru,2017/04/08,M.Chinnaswamy Stadium,Royal Challengers Bangalore,bat,Royal Challengers Bangalore,Royal Challengers Bangalore,Delhi Daredevils


In [4]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 1.1 MB/s eta 0:00:09
   --- ------------------------------------ 0.8/9.7 MB 1.2 MB/s eta 0:00:08
   ---- ----------------------------------- 1.0/9.7 MB 1.4 MB/s eta 0:00:07
   ----- ---------------------------------- 1.3/9.7 MB 1.3 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.7 MB 1.4 MB/s eta 0:00:06
   -------- ------------------------------- 2.1/9.7 MB 1.5 MB/s eta 0:00:06
   --------- ------------------------------ 2.4/9.7 MB 1.5 MB/s eta 0:00:05
   ----------- ---------------------------- 2.9/9.7 MB 1.5 MB/s eta 0:00:05
   ------------ -----------------------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
# 1. Standardize Team Names so the math doesn't get confused
team_name_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Deccan Chargers': 'Sunrisers Hyderabad', # Historically replaced by SRH
    'Gujarat Lions': 'Gujarat Titans', # Grouping newer/older regional teams if desired, but let's keep it accurate
    'Pune Warriors': 'Rising Pune Supergiant',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

# Apply the mapping to all relevant columns
columns_to_update = ['Team_A', 'Team_B', 'Toss_Winner', 'Winner']
for col in columns_to_update:
    matches_df[col] = matches_df[col].replace(team_name_mapping)

# 2. Filter out matches with "No Result" (rain washouts don't help us predict)
matches_df = matches_df[matches_df['Winner'] != 'No Result']

# 3. Create the Target Variable (1 if Team_A wins, 0 if Team_B wins)
# This is exactly what the XGBoost model will try to predict!
matches_df['Target'] = (matches_df['Winner'] == matches_df['Team_A']).astype(int)

# 4. Create a "Toss Advantage" feature (1 if Team_A won the toss, 0 otherwise)
matches_df['Team_A_Won_Toss'] = (matches_df['Toss_Winner'] == matches_df['Team_A']).astype(int)

print("Data Cleaning & Basic Features Complete!")
print(f"Remaining playable matches: {len(matches_df)}")
display(matches_df[['Team_A', 'Team_B', 'Toss_Winner', 'Winner', 'Target', 'Team_A_Won_Toss']].head())

Data Cleaning & Basic Features Complete!
Remaining playable matches: 1146


,Team_A,Team_B,Toss_Winner,Winner,Target,Team_A_Won_Toss
0,Sunrisers Hyderabad,Royal Challengers Bengaluru,Royal Challengers Bengaluru,Sunrisers Hyderabad,1,0
1,Rising Pune Supergiant,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,1,1
2,Gujarat Titans,Kolkata Knight Riders,Kolkata Knight Riders,Kolkata Knight Riders,0,0
3,Punjab Kings,Rising Pune Supergiant,Punjab Kings,Punjab Kings,1,1
4,Royal Challengers Bengaluru,Delhi Capitals,Royal Challengers Bengaluru,Royal Challengers Bengaluru,1,1


In [7]:
# 1. Calculate Overall Team Win Rates (The "Form" factor)
# How often does this team win generally?
team_wins = matches_df['Winner'].value_counts()
team_matches = matches_df['Team_A'].value_counts() + matches_df['Team_B'].value_counts()
win_rates = (team_wins / team_matches).fillna(0.5) # Default to 50% if data is missing

matches_df['Team_A_WinRate'] = matches_df['Team_A'].map(win_rates)
matches_df['Team_B_WinRate'] = matches_df['Team_B'].map(win_rates)

# 2. Calculate Head-to-Head Win Percentage (The Rivalry factor)
# Out of all matches between A and B, how often does A win?
def calc_h2h(row):
    team_a = row['Team_A']
    team_b = row['Team_B']
    
    # Find all historical matches where these two teams played each other
    matchups = matches_df[((matches_df['Team_A'] == team_a) & (matches_df['Team_B'] == team_b)) | 
                          ((matches_df['Team_A'] == team_b) & (matches_df['Team_B'] == team_a))]
    
    # If they have never played, it's a 50/50 toss-up
    if len(matchups) == 0:
        return 0.5 
        
    team_a_wins = len(matchups[matchups['Winner'] == team_a])
    return team_a_wins / len(matchups)

# Apply this calculation to every single row in our dataset
matches_df['Team_A_H2H_WinRate'] = matches_df.apply(calc_h2h, axis=1)

print("Advanced Features Engineered Successfully!")
display(matches_df[['Team_A', 'Team_B', 'Team_A_WinRate', 'Team_B_WinRate', 'Team_A_H2H_WinRate', 'Target']].head())

Advanced Features Engineered Successfully!


,Team_A,Team_B,Team_A_WinRate,Team_B_WinRate,Team_A_H2H_WinRate,Target
0,Sunrisers Hyderabad,Royal Challengers Bengaluru,0.458647,0.500000,0.542857,1
1,Rising Pune Supergiant,Mumbai Indians,0.360000,0.553114,0.416667,1
2,Gujarat Titans,Kolkata Knight Riders,0.561798,0.521236,0.750000,0
3,Punjab Kings,Rising Pune Supergiant,0.461240,0.360000,0.500000,1
4,Royal Challengers Bengaluru,Delhi Capitals,0.500000,0.457364,0.612903,1


In [8]:
!pip install xgboost scikit-learn joblib

Defaulting to user installation because normal site-packages is not writeable
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 1.7 MB/s eta 0:01:01
   ---------------------------------------- 0.8/101.7 MB 1.4 MB/s eta 0:01:13
   ---------------------------------------- 1.0/101.7 MB 1.6 MB/s eta 0:01:04
    --------------------------------------- 1.6/101.7 MB 1.6 MB/s eta 0:01:04
    --------------------------------------- 2.1/101.7 MB 1.6 MB/s eta 0:01:02
    --------------------------------------- 2.4/101.7 MB 1.7 MB/s eta 0:01:00
   - -------------------------------------- 2.9/101.7 MB 1.7 MB/s eta 0:00:57
   - -------------------------------------- 3.4/101.7 MB 1.8 MB/s eta 0:00:54
   - -------------------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

# 1. Define our Features (the math) and our Target (the answer)
features = ['Team_A_WinRate', 'Team_B_WinRate', 'Team_A_H2H_WinRate', 'Team_A_Won_Toss']
X = matches_df[features]
y = matches_df['Target']

# 2. Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize the XGBoost Model (The Engine)
print("Booting up XGBoost and training on historical IPL data...")
model = xgb.XGBClassifier(eval_metric='logloss')

# 4. Train the model!
model.fit(X_train, y_train)

# 5. Test the model's accuracy on the 20% of data it hasn't seen yet
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\n✅ Model Training Complete!")
print(f"🎯 AI Prediction Accuracy: {accuracy * 100:.2f}%")

# 6. Export the brain to a file so your React/Node.js app can use it
joblib.dump(model, 'ipl_xgboost_model.pkl')
print("💾 Saved model as 'ipl_xgboost_model.pkl' in your folder!")

Booting up XGBoost and training on historical IPL data...

✅ Model Training Complete!
🎯 AI Prediction Accuracy: 52.61%
💾 Saved model as 'ipl_xgboost_model.pkl' in your folder!


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# Upgrade 1: Give the AI 'Venue' and 'Toss Decision' Context
# We convert the text of stadiums and decisions into numerical codes
matches_df['Venue_Code'] = matches_df['Venue'].astype('category').cat.codes
matches_df['Toss_Decision_Code'] = (matches_df['Toss_Decision'] == 'bat').astype(int)

# Upgrade 2: Expanded Feature Set
features = ['Team_A_WinRate', 'Team_B_WinRate', 'Team_A_H2H_WinRate', 'Team_A_Won_Toss', 'Venue_Code', 'Toss_Decision_Code']

X = matches_df[features]
y = matches_df['Target']

# Split the data again
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Booting up Tuned Random Forest Engine...")

# Upgrade 3: A Tuned Random Forest
# n_estimators = 200 means we are building 200 "decision trees" and letting them vote on the winner.
# max_depth = 10 stops the AI from overthinking and memorizing the data.
model = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, random_state=42)

# Train it!
model.fit(X_train, y_train)

# Test it!
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\n✅ Upgraded Model Training Complete!")
print(f"🎯 New AI Prediction Accuracy: {accuracy * 100:.2f}%")

# Save the better brain
joblib.dump(model, 'ipl_rf_model.pkl')
print("💾 Saved upgraded model as 'ipl_rf_model.pkl'!")

Booting up Tuned Random Forest Engine...

✅ Upgraded Model Training Complete!
🎯 New AI Prediction Accuracy: 52.61%
💾 Saved upgraded model as 'ipl_rf_model.pkl'!


In [11]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib

# We will use the full dataset, but we are making it much smarter
df = matches_df.copy()

print("Encoding Team and Venue Identities...")

# 1. Encode Categorical Variables (Giving Teams and Venues unique IDs)
le_team = LabelEncoder()
# Combine all teams so the IDs match perfectly across all columns
all_teams = pd.concat([df['Team_A'], df['Team_B'], df['Toss_Winner']]).unique()
le_team.fit(all_teams)

df['Team_A_Code'] = le_team.transform(df['Team_A'])
df['Team_B_Code'] = le_team.transform(df['Team_B'])
df['Toss_Winner_Code'] = le_team.transform(df['Toss_Winner'])

le_venue = LabelEncoder()
df['Venue_Code'] = le_venue.fit_transform(df['Venue'].astype(str))

df['Toss_Decision_Code'] = (df['Toss_Decision'] == 'bat').astype(int)

# 2. The Final "Elite" Feature Set
features = [
    'Team_A_Code', 'Team_B_Code', 'Venue_Code', 
    'Toss_Winner_Code', 'Toss_Decision_Code',
    'Team_A_WinRate', 'Team_B_WinRate', 'Team_A_H2H_WinRate',
    'Team_A_Won_Toss'
]

X = df[features]
y = df['Target']

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Booting up 500-Tree XGBoost Engine...")

# 4. The "Secret Sauce" Hyperparameters
# colsample_bytree and subsample = 0.8 prevents the AI from memorizing the data
# learning_rate = 0.01 forces it to learn very slowly and carefully over 500 passes
model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\n🚀 Optimized Engine Training Complete!")
print(f"🎯 Elite Prediction Accuracy: {accuracy * 100:.2f}%")

# Export this specific model and the Encoders (we will need the encoders for the backend!)
joblib.dump(model, 'ipl_elite_xgboost.pkl')
joblib.dump(le_team, 'team_encoder.pkl')
joblib.dump(le_venue, 'venue_encoder.pkl')
print("💾 Saved 'ipl_elite_xgboost.pkl' and Encoders for the Node.js API!")

Encoding Team and Venue Identities...
Booting up 500-Tree XGBoost Engine...

🚀 Optimized Engine Training Complete!
🎯 Elite Prediction Accuracy: 52.17%
💾 Saved 'ipl_elite_xgboost.pkl' and Encoders for the Node.js API!


In [12]:
import pandas as pd

# Load the massive ball-by-ball dataset
print("Loading deliveries.csv... (This might take a few seconds, it's a huge file!)")

try:
    deliveries = pd.read_csv('deliveries.csv', low_memory=False)
    
    print("--- Deliveries Data Snapshot ---")
    display(deliveries.head())

    print("\n--- Available Columns ---")
    print(deliveries.columns.tolist())
    print(f"\nTotal balls bowled in dataset: {len(deliveries)}")
    
except FileNotFoundError:
    print("❌ Couldn't find 'deliveries.csv'. Please check the exact name of the file in your folder!")

Loading deliveries.csv... (This might take a few seconds, it's a huge file!)
❌ Couldn't find 'deliveries.csv'. Please check the exact name of the file in your folder!


In [13]:
import pandas as pd
import glob
import warnings
warnings.filterwarnings('ignore')

print("🕵️ Extracting ball-by-ball player & team stats...")

# 1. Grab all the ball-by-ball files (ignoring the _info files and matches.csv)
delivery_files = [f for f in glob.glob("*.csv") if "_info" not in f and f != "matches.csv"]

team_performance = []

for file in delivery_files:
    try:
        df = pd.read_csv(file)
        
        # Failsafe: if the file doesn't have batting_team, skip it
        if 'batting_team' not in df.columns: continue
        
        # Calculate Total Runs per ball
        df['total_runs'] = df['runs_off_bat'] + df['extras']
        
        # A legal ball is one that is NOT a wide and NOT a noball
        df['wides'] = df['wides'].fillna(0)
        df['noballs'] = df['noballs'].fillna(0)
        df['is_legal_ball'] = ((df['wides'] == 0) & (df['noballs'] == 0)).astype(int)
        
        # Group by batting team
        stats = df.groupby('batting_team').agg(
            runs_scored=('total_runs', 'sum'),
            legal_balls=('is_legal_ball', 'sum')
        ).reset_index()
        
        team_performance.append(stats)
    except Exception as e:
        continue

# 2. Combine all match data
all_stats_df = pd.concat(team_performance, ignore_index=True)

# 3. Standardize Team Names (just like we did earlier)
team_name_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Pune Warriors': 'Rising Pune Supergiant',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Gujarat Lions': 'Gujarat Titans' 
}
all_stats_df['batting_team'] = all_stats_df['batting_team'].replace(team_name_mapping)

# 4. Calculate the True Historical Run Rate
team_totals = all_stats_df.groupby('batting_team').sum().reset_index()
team_totals['True_Run_Rate'] = (team_totals['runs_scored'] / team_totals['legal_balls']) * 6

print("\n✅ Power Stats Calculated!")
display(team_totals.sort_values(by='True_Run_Rate', ascending=False)[['batting_team', 'True_Run_Rate']].head(10))

🕵️ Extracting ball-by-ball player & team stats...

✅ Power Stats Calculated!


,batting_team,True_Run_Rate
5,Lucknow Super Giants,9.034402
2,Gujarat Titans,8.905837
10,Royal Challengers Bengaluru,8.489456
7,Punjab Kings,8.465732
6,Mumbai Indians,8.438442
0,Chennai Super Kings,8.413110
4,Kolkata Knight Riders,8.351507
8,Rajasthan Royals,8.323716
1,Delhi Capitals,8.253093
11,Sunrisers Hyderabad,8.250335


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
import joblib

# 1. Create a dictionary of the Run Rates we just calculated
run_rate_dict = dict(zip(team_totals['batting_team'], team_totals['True_Run_Rate']))

# 2. Map these Run Rates to Team A and Team B in our match dataset
matches_df['Team_A_RunRate'] = matches_df['Team_A'].map(run_rate_dict)
matches_df['Team_B_RunRate'] = matches_df['Team_B'].map(run_rate_dict)

# 3. Calculate the "Power Diff" - Who has the stronger batting lineup?
matches_df['Run_Rate_Difference'] = matches_df['Team_A_RunRate'] - matches_df['Team_B_RunRate']

# 4. Our New, Ultimate Feature Set
features = [
    'Team_A_WinRate', 'Team_B_WinRate', 
    'Team_A_H2H_WinRate', 'Team_A_Won_Toss',
    'Team_A_RunRate', 'Team_B_RunRate', 'Run_Rate_Difference'
]

# Drop any rows where we might have missed a run rate
final_df = matches_df.dropna(subset=features)

X = final_df[features]
y = final_df['Target']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Boot up XGBoost
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\n🚀 Data Science Deep-Dive Complete!")
print(f"🎯 Final AI Prediction Accuracy: {accuracy * 100:.2f}%")

joblib.dump(model, 'ipl_final_xgboost.pkl')


🚀 Data Science Deep-Dive Complete!
🎯 Final AI Prediction Accuracy: 51.30%


['ipl_final_xgboost.pkl']

In [15]:
import glob
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score


# -----------------------------
# 1) Build per-match team run rate from ball-by-ball files
# -----------------------------
def build_team_match_run_rates():
    delivery_files = [
        f for f in glob.glob("*.csv")
        if "_info" not in f and Path(f).stem.isdigit() and f != "matches.csv"
    ]

    records = []
    for file in delivery_files:
        match_id = Path(file).stem
        try:
            cols_needed = ["batting_team", "runs_off_bat", "extras", "wides", "noballs"]
            df = pd.read_csv(file, usecols=cols_needed)
        except Exception:
            continue

        df["wides"] = df["wides"].fillna(0)
        df["noballs"] = df["noballs"].fillna(0)
        df["legal_ball"] = ((df["wides"] == 0) & (df["noballs"] == 0)).astype(int)
        df["total_runs"] = df["runs_off_bat"] + df["extras"]

        g = df.groupby("batting_team", as_index=False).agg(
            runs=("total_runs", "sum"),
            legal_balls=("legal_ball", "sum")
        )
        g = g[g["legal_balls"] > 0]
        g["rpo"] = (g["runs"] / g["legal_balls"]) * 6

        for row in g.itertuples(index=False):
            records.append((str(match_id), row.batting_team, float(row.rpo)))

    rr_df = pd.DataFrame(records, columns=["Match_ID", "Team", "Match_RunRate"])
    rr_map = {(r.Match_ID, r.Team): r.Match_RunRate for r in rr_df.itertuples(index=False)}
    return rr_df, rr_map


# -----------------------------
# 2) Time-aware feature engineering (no future leakage)
# -----------------------------
def mean_or_default(values, default):
    return float(np.mean(values)) if len(values) > 0 else default


def ema_or_default(values, span, default):
    if len(values) == 0:
        return default
    s = pd.Series(list(values), dtype="float64")
    return float(s.ewm(span=span, adjust=False).mean().iloc[-1])


def streak_from_results(values):
    # Positive streak = consecutive wins, negative = consecutive losses
    if len(values) == 0:
        return 0
    streak = 0
    last = values[-1]
    for v in reversed(values):
        if v == last:
            streak += 1
        else:
            break
    return streak if last == 1 else -streak


def h2h_winrate_for_a(h2h_winners, team_a, default=0.5):
    if len(h2h_winners) == 0:
        return default
    wins = sum(1 for w in h2h_winners if w == team_a)
    return wins / len(h2h_winners)


def build_temporal_features(matches_df, rr_map, lookback=15, h2h_lookback=20, elo_k=24, season_regress=0.30):
    df = matches_df.copy()
    df["Match_ID"] = df["Match_ID"].astype(str)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Keep only rows with usable outcomes and date
    df = df.dropna(subset=["Date"]).copy()
    df = df[df["Winner"].isin(df[["Team_A", "Team_B"]].stack().unique())].copy()

    # Strict chronological order
    df = df.sort_values(["Date", "Match_ID"]).reset_index(drop=True)

    # Stateful histories
    team_results = defaultdict(lambda: deque(maxlen=lookback))
    team_rr = defaultdict(lambda: deque(maxlen=lookback))
    team_last_date = {}

    team_elo = defaultdict(lambda: 1500.0)
    team_last_year = {}

    h2h_results = defaultdict(lambda: deque(maxlen=h2h_lookback))
    venue_results = defaultdict(lambda: deque(maxlen=10))

    toss_win_form = defaultdict(lambda: deque(maxlen=lookback))
    toss_lose_form = defaultdict(lambda: deque(maxlen=lookback))

    global_rr_default = 8.0
    rows = []

    for r in df.itertuples(index=False):
        match_id = r.Match_ID
        date = r.Date
        year = int(date.year)
        team_a = r.Team_A
        team_b = r.Team_B
        venue = r.Venue
        winner = r.Winner
        a_won_toss = int(r.Team_A_Won_Toss)

        # Season boundary Elo regression handles roster resets/mega auctions
        for t in (team_a, team_b):
            if t in team_last_year and team_last_year[t] != year:
                team_elo[t] = (1 - season_regress) * team_elo[t] + season_regress * 1500.0
            team_last_year[t] = year

        # Pre-match form features
        a_wr_l15 = mean_or_default(team_results[team_a], 0.5)
        b_wr_l15 = mean_or_default(team_results[team_b], 0.5)

        a_wr_ema = ema_or_default(team_results[team_a], span=8, default=0.5)
        b_wr_ema = ema_or_default(team_results[team_b], span=8, default=0.5)

        a_rr_l15 = mean_or_default(team_rr[team_a], global_rr_default)
        b_rr_l15 = mean_or_default(team_rr[team_b], global_rr_default)

        a_rr_ema = ema_or_default(team_rr[team_a], span=8, default=global_rr_default)
        b_rr_ema = ema_or_default(team_rr[team_b], span=8, default=global_rr_default)

        a_streak = streak_from_results(team_results[team_a])
        b_streak = streak_from_results(team_results[team_b])

        # H2H (decayed by recent window)
        pair = tuple(sorted([team_a, team_b]))
        a_h2h_recent = h2h_winrate_for_a(h2h_results[pair], team_a, default=0.5)

        # Venue familiarity
        a_venue_wr = mean_or_default(venue_results[(team_a, venue)], 0.5)
        b_venue_wr = mean_or_default(venue_results[(team_b, venue)], 0.5)

        # Rest/fatigue proxy
        a_rest_days = (date - team_last_date[team_a]).days if team_a in team_last_date else 7
        b_rest_days = (date - team_last_date[team_b]).days if team_b in team_last_date else 7

        # Toss-conditional form
        a_toss_form = mean_or_default(toss_win_form[team_a], 0.5) if a_won_toss == 1 else mean_or_default(toss_lose_form[team_a], 0.5)
        b_toss_form = mean_or_default(toss_lose_form[team_b], 0.5) if a_won_toss == 1 else mean_or_default(toss_win_form[team_b], 0.5)

        # Elo
        elo_a = float(team_elo[team_a])
        elo_b = float(team_elo[team_b])
        exp_a = 1.0 / (1.0 + 10.0 ** ((elo_b - elo_a) / 400.0))

        rows.append({
            "Match_ID": match_id,
            "Date": date,
            "Team_A": team_a,
            "Team_B": team_b,
            "Venue": venue,
            "Target": int(winner == team_a),

            "A_WR_L15": a_wr_l15,
            "B_WR_L15": b_wr_l15,
            "WR_L15_Diff": a_wr_l15 - b_wr_l15,

            "A_WR_EMA": a_wr_ema,
            "B_WR_EMA": b_wr_ema,
            "WR_EMA_Diff": a_wr_ema - b_wr_ema,

            "A_RR_L15": a_rr_l15,
            "B_RR_L15": b_rr_l15,
            "RR_L15_Diff": a_rr_l15 - b_rr_l15,

            "A_RR_EMA": a_rr_ema,
            "B_RR_EMA": b_rr_ema,
            "RR_EMA_Diff": a_rr_ema - b_rr_ema,

            "A_Streak": a_streak,
            "B_Streak": b_streak,
            "Streak_Diff": a_streak - b_streak,

            "A_H2H_Recent": a_h2h_recent,
            "A_Venue_WR": a_venue_wr,
            "B_Venue_WR": b_venue_wr,
            "Venue_WR_Diff": a_venue_wr - b_venue_wr,

            "A_Rest_Days": a_rest_days,
            "B_Rest_Days": b_rest_days,
            "Rest_Diff": a_rest_days - b_rest_days,

            "A_Toss_Form": a_toss_form,
            "B_Toss_Form": b_toss_form,
            "Toss_Form_Diff": a_toss_form - b_toss_form,

            "Elo_A": elo_a,
            "Elo_B": elo_b,
            "Elo_Diff": elo_a - elo_b,
            "Elo_Expected_A": exp_a,
            "Team_A_Won_Toss": a_won_toss,
            "Toss_Decision_Code": int(r.Toss_Decision_Code) if "Toss_Decision_Code" in df.columns else 0
        })

        # ---- Update states after feature snapshot ----
        y_a = 1 if winner == team_a else 0

        # Elo update
        team_elo[team_a] = elo_a + elo_k * (y_a - exp_a)
        team_elo[team_b] = elo_b + elo_k * ((1 - y_a) - (1 - exp_a))

        # Win/loss form
        team_results[team_a].append(y_a)
        team_results[team_b].append(1 - y_a)

        # H2H and venue form
        h2h_results[pair].append(winner)
        venue_results[(team_a, venue)].append(y_a)
        venue_results[(team_b, venue)].append(1 - y_a)

        # Toss-conditioned form
        if a_won_toss == 1:
            toss_win_form[team_a].append(y_a)
            toss_lose_form[team_b].append(1 - y_a)
        else:
            toss_lose_form[team_a].append(y_a)
            toss_win_form[team_b].append(1 - y_a)

        # Recent run-rate form (derived from this completed match)
        a_match_rr = rr_map.get((match_id, team_a), np.nan)
        b_match_rr = rr_map.get((match_id, team_b), np.nan)
        if pd.notna(a_match_rr):
            team_rr[team_a].append(float(a_match_rr))
        if pd.notna(b_match_rr):
            team_rr[team_b].append(float(b_match_rr))

        team_last_date[team_a] = date
        team_last_date[team_b] = date

    return pd.DataFrame(rows)


# -----------------------------
# 3) Build transformed dataset + chronological train/test
# -----------------------------
rr_df, rr_map = build_team_match_run_rates()
temporal_df = build_temporal_features(matches_df, rr_map, lookback=15, h2h_lookback=20, elo_k=24, season_regress=0.30)

# Time-safe split: train on <= 2022, test on >= 2023 (adjust if needed)
train_df = temporal_df[temporal_df["Date"] < "2023-01-01"].copy()
test_df = temporal_df[temporal_df["Date"] >= "2023-01-01"].copy()

if len(test_df) < 50:
    # fallback to last 20% chronologically
    cut = int(len(temporal_df) * 0.8)
    train_df = temporal_df.iloc[:cut].copy()
    test_df = temporal_df.iloc[cut:].copy()

feature_cols = [
    "A_WR_L15", "B_WR_L15", "WR_L15_Diff",
    "A_WR_EMA", "B_WR_EMA", "WR_EMA_Diff",
    "A_RR_L15", "B_RR_L15", "RR_L15_Diff",
    "A_RR_EMA", "B_RR_EMA", "RR_EMA_Diff",
    "A_Streak", "B_Streak", "Streak_Diff",
    "A_H2H_Recent",
    "A_Venue_WR", "B_Venue_WR", "Venue_WR_Diff",
    "A_Rest_Days", "B_Rest_Days", "Rest_Diff",
    "A_Toss_Form", "B_Toss_Form", "Toss_Form_Diff",
    "Elo_A", "Elo_B", "Elo_Diff", "Elo_Expected_A",
    "Team_A_Won_Toss", "Toss_Decision_Code"
]

X_train = train_df[feature_cols]
y_train = train_df["Target"]
X_test = test_df[feature_cols]
y_test = test_df["Target"]

model = xgb.XGBClassifier(
    n_estimators=700,
    max_depth=4,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    min_child_weight=2,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.50).astype(int)

acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba)

print(f"Temporal feature rows: {len(temporal_df)}")
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"Time-aware XGBoost Accuracy: {acc * 100:.2f}%")
print(f"Time-aware XGBoost ROC-AUC: {auc:.4f}")

# Reusable transformed frame for your next experiments
advanced_matches_df = temporal_df.copy()

Temporal feature rows: 1146
Train rows: 932 | Test rows: 214
Time-aware XGBoost Accuracy: 42.52%
Time-aware XGBoost ROC-AUC: 0.4064


In [16]:
print('Current temporal model metrics from last run:')
if 'acc' in globals() and 'auc' in globals():
    print(f"Accuracy: {acc * 100:.2f}%")
    print(f"ROC-AUC: {auc:.4f}")
else:
    print('acc/auc not found in kernel. Run Cell 12 first.')

Current temporal model metrics from last run:
Accuracy: 42.52%
ROC-AUC: 0.4064


In [17]:
from sklearn.model_selection import ParameterSampler


# -----------------------------
# 4) Accuracy booster: symmetry augmentation + tuned threshold
# -----------------------------
def build_swapped_training_rows(df):
    z = df.copy()

    # Swap team-perspective columns
    pair_swaps = [
        ("A_WR_L15", "B_WR_L15"),
        ("A_WR_EMA", "B_WR_EMA"),
        ("A_RR_L15", "B_RR_L15"),
        ("A_RR_EMA", "B_RR_EMA"),
        ("A_Streak", "B_Streak"),
        ("A_Venue_WR", "B_Venue_WR"),
        ("A_Rest_Days", "B_Rest_Days"),
        ("A_Toss_Form", "B_Toss_Form"),
        ("Elo_A", "Elo_B")
    ]

    for a_col, b_col in pair_swaps:
        if a_col in z.columns and b_col in z.columns:
            z[a_col], z[b_col] = z[b_col].values, z[a_col].values

    # Any "Diff" feature should change sign under team swap
    diff_cols = [c for c in z.columns if c.endswith("_Diff")]
    for c in diff_cols:
        z[c] = -z[c]

    # Team-A oriented probabilities/features invert under swap
    if "A_H2H_Recent" in z.columns:
        z["A_H2H_Recent"] = 1.0 - z["A_H2H_Recent"]
    if "Elo_Expected_A" in z.columns:
        z["Elo_Expected_A"] = 1.0 - z["Elo_Expected_A"]

    # Toss winner relation to Team_A flips when teams are swapped
    if "Team_A_Won_Toss" in z.columns:
        z["Team_A_Won_Toss"] = 1 - z["Team_A_Won_Toss"]

    # Target also flips
    z["Target"] = 1 - z["Target"]
    return z


feature_cols_v2 = [
    "A_WR_L15", "B_WR_L15", "WR_L15_Diff",
    "A_WR_EMA", "B_WR_EMA", "WR_EMA_Diff",
    "A_RR_L15", "B_RR_L15", "RR_L15_Diff",
    "A_RR_EMA", "B_RR_EMA", "RR_EMA_Diff",
    "A_Streak", "B_Streak", "Streak_Diff",
    "A_H2H_Recent",
    "A_Venue_WR", "B_Venue_WR", "Venue_WR_Diff",
    "A_Rest_Days", "B_Rest_Days", "Rest_Diff",
    "A_Toss_Form", "B_Toss_Form", "Toss_Form_Diff",
    "Elo_A", "Elo_B", "Elo_Diff", "Elo_Expected_A",
    "Team_A_Won_Toss", "Toss_Decision_Code"
]

# Base chronological split
train_base = temporal_df[temporal_df["Date"] < "2023-01-01"].copy()
test_holdout = temporal_df[temporal_df["Date"] >= "2023-01-01"].copy()

if len(test_holdout) < 50:
    cut = int(len(temporal_df) * 0.8)
    train_base = temporal_df.iloc[:cut].copy()
    test_holdout = temporal_df.iloc[cut:].copy()

# Chronological train/validation split within training period
val_cut = int(len(train_base) * 0.85)
train_part = train_base.iloc[:val_cut].copy()
val_part = train_base.iloc[val_cut:].copy()

# Symmetry augmentation only on train_part
train_swapped = build_swapped_training_rows(train_part)
train_aug = pd.concat([train_part, train_swapped], ignore_index=True)

X_tr = train_aug[feature_cols_v2]
y_tr = train_aug["Target"]
X_val = val_part[feature_cols_v2]
y_val = val_part["Target"]
X_te = test_holdout[feature_cols_v2]
y_te = test_holdout["Target"]

param_grid = {
    "n_estimators": [300, 500, 700, 900],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "subsample": [0.75, 0.85, 0.95],
    "colsample_bytree": [0.75, 0.85, 0.95],
    "min_child_weight": [1, 2, 4, 6],
    "reg_lambda": [1.0, 2.0, 4.0, 8.0],
    "gamma": [0.0, 0.5, 1.0]
}

search_space = list(ParameterSampler(param_grid, n_iter=24, random_state=42))

best = None
for p in search_space:
    m = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        **p
    )
    m.fit(X_tr, y_tr)

    val_proba = m.predict_proba(X_val)[:, 1]

    # Direction correction if the model is consistently inverted
    val_auc_raw = roc_auc_score(y_val, val_proba)
    if val_auc_raw < 0.5:
        val_proba = 1 - val_proba

    # Threshold tuning for validation accuracy
    thresholds = np.arange(0.35, 0.66, 0.01)
    scores = []
    for t in thresholds:
        val_pred_t = (val_proba >= t).astype(int)
        scores.append(accuracy_score(y_val, val_pred_t))

    best_idx = int(np.argmax(scores))
    val_acc_best = float(scores[best_idx])
    best_t = float(thresholds[best_idx])

    if best is None or val_acc_best > best["val_acc"]:
        best = {
            "model": m,
            "params": p,
            "val_acc": val_acc_best,
            "val_threshold": best_t,
            "invert": bool(val_auc_raw < 0.5)
        }

# Retrain best model on full pre-2023 set + its swapped mirror
train_full_swapped = build_swapped_training_rows(train_base)
train_full_aug = pd.concat([train_base, train_full_swapped], ignore_index=True)

X_train_full = train_full_aug[feature_cols_v2]
y_train_full = train_full_aug["Target"]

best_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    **best["params"]
)
best_model.fit(X_train_full, y_train_full)

te_proba = best_model.predict_proba(X_te)[:, 1]
if best["invert"]:
    te_proba = 1 - te_proba

te_pred = (te_proba >= best["val_threshold"]).astype(int)

acc_v2 = accuracy_score(y_te, te_pred)
auc_v2 = roc_auc_score(y_te, te_proba)

print("\n=== BOOSTED TEMPORAL MODEL RESULTS ===")
print(f"Train rows (base): {len(train_base)} | Train rows (aug): {len(train_full_aug)}")
print(f"Holdout rows: {len(test_holdout)}")
print(f"Best validation threshold: {best['val_threshold']:.2f}")
print(f"Best params: {best['params']}")
print(f"Boosted Accuracy: {acc_v2 * 100:.2f}%")
print(f"Boosted ROC-AUC: {auc_v2:.4f}")

# Keep handy for downstream notebook steps
boosted_model = best_model
boosted_feature_cols = feature_cols_v2
boosted_holdout_pred = te_pred
boosted_holdout_proba = te_proba


=== BOOSTED TEMPORAL MODEL RESULTS ===
Train rows (base): 932 | Train rows (aug): 1864
Holdout rows: 214
Best validation threshold: 0.57
Best params: {'subsample': 0.75, 'reg_lambda': 4.0, 'n_estimators': 500, 'min_child_weight': 2, 'max_depth': 6, 'learning_rate': 0.03, 'gamma': 1.0, 'colsample_bytree': 0.75}
Boosted Accuracy: 46.73%
Boosted ROC-AUC: 0.4489


In [18]:
import csv


# -----------------------------
# 5) Rebuild clean match table from raw *_info files (no franchise merging)
# -----------------------------
def build_raw_matches_df_from_info():
    all_info_files = sorted(glob.glob("*_info.csv"))
    rows = []

    for file in all_info_files:
        match_id = Path(file).stem.replace("_info", "")
        info = {
            "Match_ID": match_id,
            "Date": None,
            "Venue": "Unknown",
            "Toss_Winner": None,
            "Toss_Decision": None,
            "Winner": None,
            "Team_A": None,
            "Team_B": None,
        }
        teams = []

        try:
            with open(file, "r", encoding="utf-8") as f:
                reader = csv.reader(f)
                for r in reader:
                    if len(r) < 3:
                        continue
                    key = r[1]
                    val = r[2]

                    if key == "date":
                        info["Date"] = val
                    elif key == "venue":
                        info["Venue"] = val
                    elif key == "toss_winner":
                        info["Toss_Winner"] = val
                    elif key == "toss_decision":
                        info["Toss_Decision"] = val
                    elif key == "winner":
                        info["Winner"] = val
                    elif key == "team":
                        teams.append(val)
        except Exception:
            continue

        if len(teams) >= 2:
            info["Team_A"] = teams[0]
            info["Team_B"] = teams[1]
        else:
            continue

        # Keep only valid decided matches
        if info["Winner"] not in (info["Team_A"], info["Team_B"]):
            continue

        info["Toss_Decision_Code"] = 1 if str(info["Toss_Decision"]).lower() == "bat" else 0
        info["Team_A_Won_Toss"] = int(info["Toss_Winner"] == info["Team_A"])

        rows.append(info)

    m = pd.DataFrame(rows)
    m["Date"] = pd.to_datetime(m["Date"], errors="coerce")
    m = m.dropna(subset=["Date"]).sort_values(["Date", "Match_ID"]).reset_index(drop=True)
    m["Target"] = (m["Winner"] == m["Team_A"]).astype(int)
    return m


raw_matches_df = build_raw_matches_df_from_info()
rr_df2, rr_map2 = build_team_match_run_rates()
temporal_raw_df = build_temporal_features(raw_matches_df, rr_map2, lookback=15, h2h_lookback=20, elo_k=24, season_regress=0.35)

train_raw = temporal_raw_df[temporal_raw_df["Date"] < "2023-01-01"].copy()
test_raw = temporal_raw_df[temporal_raw_df["Date"] >= "2023-01-01"].copy()
if len(test_raw) < 50:
    cut = int(len(temporal_raw_df) * 0.8)
    train_raw = temporal_raw_df.iloc[:cut].copy()
    test_raw = temporal_raw_df.iloc[cut:].copy()

val_cut = int(len(train_raw) * 0.85)
tr_raw = train_raw.iloc[:val_cut].copy()
va_raw = train_raw.iloc[val_cut:].copy()

tr_raw_aug = pd.concat([tr_raw, build_swapped_training_rows(tr_raw)], ignore_index=True)

Xtr = tr_raw_aug[feature_cols_v2]
ytr = tr_raw_aug["Target"]
Xva = va_raw[feature_cols_v2]
yva = va_raw["Target"]
Xte = test_raw[feature_cols_v2]
yte = test_raw["Target"]

model_raw = xgb.XGBClassifier(
    n_estimators=700,
    max_depth=5,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    reg_lambda=4,
    gamma=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)
model_raw.fit(Xtr, ytr)

va_proba = model_raw.predict_proba(Xva)[:, 1]
if roc_auc_score(yva, va_proba) < 0.5:
    va_proba = 1 - va_proba

cand_t = np.arange(0.35, 0.66, 0.01)
va_accs = [accuracy_score(yva, (va_proba >= t).astype(int)) for t in cand_t]
best_t2 = float(cand_t[int(np.argmax(va_accs))])

# Refit on full raw train + swapped
tf_aug = pd.concat([train_raw, build_swapped_training_rows(train_raw)], ignore_index=True)
model_raw.fit(tf_aug[feature_cols_v2], tf_aug["Target"])

te_proba = model_raw.predict_proba(Xte)[:, 1]
if roc_auc_score(yva, model_raw.predict_proba(Xva)[:, 1]) < 0.5:
    te_proba = 1 - te_proba

te_pred = (te_proba >= best_t2).astype(int)
acc_raw = accuracy_score(yte, te_pred)
auc_raw = roc_auc_score(yte, te_proba)

print("\n=== RAW-NAMES TEMPORAL MODEL (NO FRANCHISE MERGE) ===")
print(f"Rows: {len(temporal_raw_df)} | Train: {len(train_raw)} | Test: {len(test_raw)}")
print(f"Best threshold: {best_t2:.2f}")
print(f"Accuracy: {acc_raw * 100:.2f}%")
print(f"ROC-AUC: {auc_raw:.4f}")


=== RAW-NAMES TEMPORAL MODEL (NO FRANCHISE MERGE) ===
Rows: 1146 | Train: 932 | Test: 214
Best threshold: 0.49
Accuracy: 49.53%
ROC-AUC: 0.4733


In [19]:
# -----------------------------
# 6) Recency-weighted model + team/venue identity codes
# -----------------------------
work_df = temporal_raw_df.copy().sort_values(["Date", "Match_ID"]).reset_index(drop=True)

# Focus on modern IPL era for training to reduce stale signal
train_mod = work_df[(work_df["Date"] >= "2017-01-01") & (work_df["Date"] < "2023-01-01")].copy()
test_mod = work_df[work_df["Date"] >= "2023-01-01"].copy()
if len(train_mod) < 300:
    train_mod = work_df[work_df["Date"] < "2023-01-01"].copy()

# Chronological validation split
val_cut2 = int(len(train_mod) * 0.85)
tr2 = train_mod.iloc[:val_cut2].copy()
va2 = train_mod.iloc[val_cut2:].copy()

# Fit encoders on training only
team_vocab = sorted(pd.concat([tr2["Team_A"], tr2["Team_B"]]).dropna().unique().tolist())
venue_vocab = sorted(tr2["Venue"].dropna().unique().tolist())
team_to_id = {t: i for i, t in enumerate(team_vocab)}
venue_to_id = {v: i for i, v in enumerate(venue_vocab)}

for d in (tr2, va2, test_mod):
    d["Team_A_Code"] = d["Team_A"].map(team_to_id).fillna(-1).astype(int)
    d["Team_B_Code"] = d["Team_B"].map(team_to_id).fillna(-1).astype(int)
    d["Venue_Code"] = d["Venue"].map(venue_to_id).fillna(-1).astype(int)

feat_v3 = feature_cols_v2 + ["Team_A_Code", "Team_B_Code", "Venue_Code"]

# Recency weights (more weight to recent matches)
min_date = tr2["Date"].min()
age_days = (tr2["Date"] - min_date).dt.days.clip(lower=0)
weights = np.exp(age_days / 730.0)  # ~2-year decay scale
weights = np.asarray(weights / np.mean(weights), dtype=float)

m3 = xgb.XGBClassifier(
    n_estimators=900,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_lambda=6,
    gamma=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)
m3.fit(tr2[feat_v3], tr2["Target"], sample_weight=weights)

va_proba3 = m3.predict_proba(va2[feat_v3])[:, 1]
invert3 = False
if roc_auc_score(va2["Target"], va_proba3) < 0.5:
    va_proba3 = 1 - va_proba3
    invert3 = True

ths = np.arange(0.30, 0.71, 0.01)
accs3 = [accuracy_score(va2["Target"], (va_proba3 >= t).astype(int)) for t in ths]
best_t3 = float(ths[int(np.argmax(accs3))])

# Retrain on full modern training period
min_date_f = train_mod["Date"].min()
age_days_f = (train_mod["Date"] - min_date_f).dt.days.clip(lower=0)
weights_f = np.exp(age_days_f / 730.0)
weights_f = np.asarray(weights_f / np.mean(weights_f), dtype=float)

for d in (train_mod, test_mod):
    d["Team_A_Code"] = d["Team_A"].map(team_to_id).fillna(-1).astype(int)
    d["Team_B_Code"] = d["Team_B"].map(team_to_id).fillna(-1).astype(int)
    d["Venue_Code"] = d["Venue"].map(venue_to_id).fillna(-1).astype(int)

m3.fit(train_mod[feat_v3], train_mod["Target"], sample_weight=weights_f)
te_proba3 = m3.predict_proba(test_mod[feat_v3])[:, 1]
if invert3:
    te_proba3 = 1 - te_proba3

test_pred3 = (te_proba3 >= best_t3).astype(int)
acc3 = accuracy_score(test_mod["Target"], test_pred3)
auc3 = roc_auc_score(test_mod["Target"], te_proba3)

print("\n=== RECENCY-WEIGHTED MODEL (MODERN ERA) ===")
print(f"Train rows: {len(train_mod)} | Test rows: {len(test_mod)}")
print(f"Best threshold: {best_t3:.2f}")
print(f"Accuracy: {acc3 * 100:.2f}%")
print(f"ROC-AUC: {auc3:.4f}")


=== RECENCY-WEIGHTED MODEL (MODERN ERA) ===
Train rows: 364 | Test rows: 214
Best threshold: 0.30
Accuracy: 50.93%
ROC-AUC: 0.5128


In [20]:
# Quick baseline check on the same holdout
if 'test_mod' in globals():
    base_majority = max(test_mod['Target'].mean(), 1 - test_mod['Target'].mean())
    print(f"Holdout Team_A win-rate: {test_mod['Target'].mean():.4f}")
    print(f"Naive majority baseline accuracy: {base_majority * 100:.2f}%")
else:
    print('Run previous cell first.')

Holdout Team_A win-rate: 0.5047
Naive majority baseline accuracy: 50.47%


In [21]:
# -----------------------------
# 7) Add cricket-specific rolling strengths from ball-by-ball files
# -----------------------------
def build_match_team_strength_map():
    files = [
        f for f in glob.glob("*.csv")
        if "_info" not in f and Path(f).stem.isdigit() and f != "matches.csv"
    ]

    out = {}
    for f in files:
        match_id = Path(f).stem
        try:
            use = ["batting_team", "bowling_team", "runs_off_bat", "extras", "wides", "noballs", "player_dismissed"]
            d = pd.read_csv(f, usecols=use)
        except Exception:
            continue

        d["wides"] = d["wides"].fillna(0)
        d["noballs"] = d["noballs"].fillna(0)
        d["legal_ball"] = ((d["wides"] == 0) & (d["noballs"] == 0)).astype(int)
        d["total_runs"] = d["runs_off_bat"] + d["extras"]
        d["is_wicket"] = d["player_dismissed"].notna().astype(int)

        bat = d.groupby("batting_team", as_index=False).agg(
            runs_scored=("total_runs", "sum"),
            legal_balls_faced=("legal_ball", "sum"),
            wickets_lost=("is_wicket", "sum")
        )
        bowl = d.groupby("bowling_team", as_index=False).agg(
            runs_conceded=("total_runs", "sum"),
            legal_balls_bowled=("legal_ball", "sum"),
            wickets_taken=("is_wicket", "sum")
        )

        m = pd.merge(bat, bowl, left_on="batting_team", right_on="bowling_team", how="outer")
        m["Team"] = m["batting_team"].fillna(m["bowling_team"])

        for c in ["runs_scored", "legal_balls_faced", "wickets_lost", "runs_conceded", "legal_balls_bowled", "wickets_taken"]:
            m[c] = m[c].fillna(0)

        m = m[m["Team"].notna()]
        for row in m.itertuples(index=False):
            lb_f = max(float(row.legal_balls_faced), 1.0)
            lb_b = max(float(row.legal_balls_bowled), 1.0)
            out[(str(match_id), row.Team)] = {
                "bat_rr": (float(row.runs_scored) / lb_f) * 6.0,
                "bowl_rr_conceded": (float(row.runs_conceded) / lb_b) * 6.0,
                "bat_wk_rate": float(row.wickets_lost) / lb_f,
                "bowl_wk_rate": float(row.wickets_taken) / lb_b,
            }

    return out


def add_rolling_strength_features(df, strength_map, span=8, lookback=15):
    z = df.copy().sort_values(["Date", "Match_ID"]).reset_index(drop=True)

    from collections import defaultdict, deque
    hist_bat_rr = defaultdict(lambda: deque(maxlen=lookback))
    hist_bowl_rr = defaultdict(lambda: deque(maxlen=lookback))
    hist_bat_wk = defaultdict(lambda: deque(maxlen=lookback))
    hist_bowl_wk = defaultdict(lambda: deque(maxlen=lookback))

    A_bat_ema, B_bat_ema = [], []
    A_bowl_ema, B_bowl_ema = [], []
    A_bwk_ema, B_bwk_ema = [], []
    A_twk_ema, B_twk_ema = [], []

    def ema_or_default_local(values, default):
        if len(values) == 0:
            return default
        return float(pd.Series(list(values), dtype="float64").ewm(span=span, adjust=False).mean().iloc[-1])

    for r in z.itertuples(index=False):
        a = r.Team_A
        b = r.Team_B

        A_bat_ema.append(ema_or_default_local(hist_bat_rr[a], 8.0))
        B_bat_ema.append(ema_or_default_local(hist_bat_rr[b], 8.0))

        A_bowl_ema.append(ema_or_default_local(hist_bowl_rr[a], 8.0))
        B_bowl_ema.append(ema_or_default_local(hist_bowl_rr[b], 8.0))

        A_bwk_ema.append(ema_or_default_local(hist_bat_wk[a], 0.04))
        B_bwk_ema.append(ema_or_default_local(hist_bat_wk[b], 0.04))

        A_twk_ema.append(ema_or_default_local(hist_bowl_wk[a], 0.04))
        B_twk_ema.append(ema_or_default_local(hist_bowl_wk[b], 0.04))

        sa = strength_map.get((str(r.Match_ID), a), None)
        sb = strength_map.get((str(r.Match_ID), b), None)
        if sa is not None:
            hist_bat_rr[a].append(sa["bat_rr"])
            hist_bowl_rr[a].append(sa["bowl_rr_conceded"])
            hist_bat_wk[a].append(sa["bat_wk_rate"])
            hist_bowl_wk[a].append(sa["bowl_wk_rate"])
        if sb is not None:
            hist_bat_rr[b].append(sb["bat_rr"])
            hist_bowl_rr[b].append(sb["bowl_rr_conceded"])
            hist_bat_wk[b].append(sb["bat_wk_rate"])
            hist_bowl_wk[b].append(sb["bowl_wk_rate"])

    z["A_BatRR_EMA"] = A_bat_ema
    z["B_BatRR_EMA"] = B_bat_ema
    z["BatRR_EMA_Diff"] = z["A_BatRR_EMA"] - z["B_BatRR_EMA"]

    z["A_BowlRRConcede_EMA"] = A_bowl_ema
    z["B_BowlRRConcede_EMA"] = B_bowl_ema
    z["BowlRRConcede_EMA_Diff"] = z["A_BowlRRConcede_EMA"] - z["B_BowlRRConcede_EMA"]

    z["A_BatWkRate_EMA"] = A_bwk_ema
    z["B_BatWkRate_EMA"] = B_bwk_ema
    z["BatWkRate_EMA_Diff"] = z["A_BatWkRate_EMA"] - z["B_BatWkRate_EMA"]

    z["A_BowlWkRate_EMA"] = A_twk_ema
    z["B_BowlWkRate_EMA"] = B_twk_ema
    z["BowlWkRate_EMA_Diff"] = z["A_BowlWkRate_EMA"] - z["B_BowlWkRate_EMA"]

    z["A_NetRR_EMA"] = z["A_BatRR_EMA"] - z["A_BowlRRConcede_EMA"]
    z["B_NetRR_EMA"] = z["B_BatRR_EMA"] - z["B_BowlRRConcede_EMA"]
    z["NetRR_EMA_Diff"] = z["A_NetRR_EMA"] - z["B_NetRR_EMA"]

    return z


strength_map = build_match_team_strength_map()
rich_df = add_rolling_strength_features(temporal_raw_df, strength_map, span=8, lookback=15)

train_rich = rich_df[rich_df["Date"] < "2023-01-01"].copy()
test_rich = rich_df[rich_df["Date"] >= "2023-01-01"].copy()

val_cut3 = int(len(train_rich) * 0.85)
tr3 = train_rich.iloc[:val_cut3].copy()
va3 = train_rich.iloc[val_cut3:].copy()

extra_feats = [
    "A_BatRR_EMA", "B_BatRR_EMA", "BatRR_EMA_Diff",
    "A_BowlRRConcede_EMA", "B_BowlRRConcede_EMA", "BowlRRConcede_EMA_Diff",
    "A_BatWkRate_EMA", "B_BatWkRate_EMA", "BatWkRate_EMA_Diff",
    "A_BowlWkRate_EMA", "B_BowlWkRate_EMA", "BowlWkRate_EMA_Diff",
    "A_NetRR_EMA", "B_NetRR_EMA", "NetRR_EMA_Diff"
]
feat_v4 = feature_cols_v2 + extra_feats

m4 = xgb.XGBClassifier(
    n_estimators=900,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_lambda=6,
    gamma=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)
m4.fit(tr3[feat_v4], tr3["Target"])

va_proba4 = m4.predict_proba(va3[feat_v4])[:, 1]
invert4 = False
if roc_auc_score(va3["Target"], va_proba4) < 0.5:
    va_proba4 = 1 - va_proba4
    invert4 = True

ths4 = np.arange(0.30, 0.71, 0.01)
accs4 = [accuracy_score(va3["Target"], (va_proba4 >= t).astype(int)) for t in ths4]
best_t4 = float(ths4[int(np.argmax(accs4))])

m4.fit(train_rich[feat_v4], train_rich["Target"])
te_proba4 = m4.predict_proba(test_rich[feat_v4])[:, 1]
if invert4:
    te_proba4 = 1 - te_proba4

te_pred4 = (te_proba4 >= best_t4).astype(int)
acc4 = accuracy_score(test_rich["Target"], te_pred4)
auc4 = roc_auc_score(test_rich["Target"], te_proba4)

print("\n=== RICH T20 FEATURES MODEL ===")
print(f"Rows train/test: {len(train_rich)}/{len(test_rich)}")
print(f"Best threshold: {best_t4:.2f}")
print(f"Accuracy: {acc4 * 100:.2f}%")
print(f"ROC-AUC: {auc4:.4f}")


=== RICH T20 FEATURES MODEL ===
Rows train/test: 932/214
Best threshold: 0.60
Accuracy: 50.00%
ROC-AUC: 0.5031


In [22]:
# -----------------------------
# 8) Walk-forward backtest (rolling retrain by season)
# -----------------------------
from sklearn.metrics import accuracy_score, roc_auc_score


# Use the richest available feature table
if "rich_df" in globals():
    wf_df = rich_df.copy()
elif "temporal_raw_df" in globals():
    wf_df = temporal_raw_df.copy()
else:
    raise RuntimeError("Need rich_df or temporal_raw_df in memory. Run the temporal feature cells first.")

wf_df = wf_df.sort_values(["Date", "Match_ID"]).reset_index(drop=True)
wf_df["Season"] = wf_df["Date"].dt.year

# Feature list: keep only columns that exist
candidate_feats = [
    "A_WR_L15", "B_WR_L15", "WR_L15_Diff",
    "A_WR_EMA", "B_WR_EMA", "WR_EMA_Diff",
    "A_RR_L15", "B_RR_L15", "RR_L15_Diff",
    "A_RR_EMA", "B_RR_EMA", "RR_EMA_Diff",
    "A_Streak", "B_Streak", "Streak_Diff",
    "A_H2H_Recent",
    "A_Venue_WR", "B_Venue_WR", "Venue_WR_Diff",
    "A_Rest_Days", "B_Rest_Days", "Rest_Diff",
    "A_Toss_Form", "B_Toss_Form", "Toss_Form_Diff",
    "Elo_A", "Elo_B", "Elo_Diff", "Elo_Expected_A",
    "Team_A_Won_Toss", "Toss_Decision_Code",
    "A_BatRR_EMA", "B_BatRR_EMA", "BatRR_EMA_Diff",
    "A_BowlRRConcede_EMA", "B_BowlRRConcede_EMA", "BowlRRConcede_EMA_Diff",
    "A_BatWkRate_EMA", "B_BatWkRate_EMA", "BatWkRate_EMA_Diff",
    "A_BowlWkRate_EMA", "B_BowlWkRate_EMA", "BowlWkRate_EMA_Diff",
    "A_NetRR_EMA", "B_NetRR_EMA", "NetRR_EMA_Diff"
]
wf_feature_cols = [c for c in candidate_feats if c in wf_df.columns]

# Seasons for walk-forward
seasons = sorted(wf_df["Season"].dropna().astype(int).unique().tolist())
results = []

# Start from a season where we have enough history (rolling 3-season train)
for test_season in seasons:
    train_seasons = [s for s in seasons if s < test_season]
    if len(train_seasons) < 3:
        continue

    rolling_train_seasons = train_seasons[-3:]
    train_fold = wf_df[wf_df["Season"].isin(rolling_train_seasons)].copy()
    test_fold = wf_df[wf_df["Season"] == test_season].copy()
    if len(train_fold) < 100 or len(test_fold) < 20:
        continue

    # Validation = latest season inside train window for threshold tuning
    val_season = max(rolling_train_seasons)
    tr_fold = train_fold[train_fold["Season"] < val_season].copy()
    va_fold = train_fold[train_fold["Season"] == val_season].copy()

    if len(tr_fold) < 80 or len(va_fold) < 20:
        continue

    # Recency weights inside train fold
    min_d = tr_fold["Date"].min()
    age_days = (tr_fold["Date"] - min_d).dt.days.clip(lower=0)
    w = np.exp(age_days / 730.0)
    w = np.asarray(w / np.mean(w), dtype=float)

    m = xgb.XGBClassifier(
        n_estimators=700,
        max_depth=5,
        learning_rate=0.02,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=2,
        reg_lambda=6,
        gamma=1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    )
    m.fit(tr_fold[wf_feature_cols], tr_fold["Target"], sample_weight=w)

    va_proba = m.predict_proba(va_fold[wf_feature_cols])[:, 1]
    invert = False
    va_auc = roc_auc_score(va_fold["Target"], va_proba)
    if va_auc < 0.5:
        va_proba = 1 - va_proba
        invert = True

    ths = np.arange(0.35, 0.66, 0.01)
    va_accs = [accuracy_score(va_fold["Target"], (va_proba >= t).astype(int)) for t in ths]
    best_t = float(ths[int(np.argmax(va_accs))])

    # Refit on full rolling train window with recency weighting
    min_d2 = train_fold["Date"].min()
    age_days2 = (train_fold["Date"] - min_d2).dt.days.clip(lower=0)
    w2 = np.exp(age_days2 / 730.0)
    w2 = np.asarray(w2 / np.mean(w2), dtype=float)

    m.fit(train_fold[wf_feature_cols], train_fold["Target"], sample_weight=w2)
    te_proba = m.predict_proba(test_fold[wf_feature_cols])[:, 1]
    if invert:
        te_proba = 1 - te_proba

    te_pred = (te_proba >= best_t).astype(int)
    te_acc = accuracy_score(test_fold["Target"], te_pred)
    te_auc = roc_auc_score(test_fold["Target"], te_proba)

    baseline = max(test_fold["Target"].mean(), 1 - test_fold["Target"].mean())
    lift = te_acc - baseline

    results.append({
        "Season": test_season,
        "Train_Seasons": rolling_train_seasons,
        "Train_Rows": len(train_fold),
        "Test_Rows": len(test_fold),
        "Threshold": round(best_t, 2),
        "Accuracy": te_acc,
        "ROC_AUC": te_auc,
        "Baseline": baseline,
        "Lift_vs_Baseline": lift
    })

wf_results = pd.DataFrame(results)

if len(wf_results) == 0:
    print("No valid walk-forward folds found. Try reducing min row constraints.")
else:
    wf_results = wf_results.sort_values("Season").reset_index(drop=True)

    weighted_acc = np.average(wf_results["Accuracy"], weights=wf_results["Test_Rows"])
    weighted_auc = np.average(wf_results["ROC_AUC"], weights=wf_results["Test_Rows"])
    weighted_base = np.average(wf_results["Baseline"], weights=wf_results["Test_Rows"])

    print("\n=== WALK-FORWARD BACKTEST (ROLLING 3-SEASON TRAIN) ===")
    display(wf_results)
    print(f"Weighted Accuracy: {weighted_acc * 100:.2f}%")
    print(f"Weighted ROC-AUC: {weighted_auc:.4f}")
    print(f"Weighted Baseline: {weighted_base * 100:.2f}%")
    print(f"Weighted Lift: {(weighted_acc - weighted_base) * 100:.2f}%")


=== WALK-FORWARD BACKTEST (ROLLING 3-SEASON TRAIN) ===


,Season,Train_Seasons,Train_Rows,Test_Rows,Threshold,Accuracy,ROC_AUC,Baseline,Lift_vs_Baseline
0,2011,"[2008, 2009, 2010]",173,72,0.53,0.638889,0.630148,0.541667,0.097222
1,2012,"[2009, 2010, 2011]",187,74,0.37,0.472973,0.453067,0.554054,-0.081081
2,2013,"[2010, 2011, 2012]",205,74,0.35,0.567568,0.555070,0.702703,-0.135135
3,2014,"[2011, 2012, 2013]",220,59,0.41,0.423729,0.482719,0.525424,-0.101695
4,2015,"[2012, 2013, 2014]",207,56,0.55,0.535714,0.501935,0.553571,-0.017857
5,2016,"[2013, 2014, 2015]",189,60,0.38,0.433333,0.457589,0.533333,-0.100000
6,2017,"[2014, 2015, 2016]",175,58,0.47,0.431034,0.425481,0.551724,-0.120690
7,2018,"[2015, 2016, 2017]",174,60,0.45,0.483333,0.507812,0.533333,-0.050000
8,2019,"[2016, 2017, 2018]",178,57,0.66,0.561404,0.470130,0.614035,-0.052632
9,2020,"[2017, 2018, 2019]",175,56,0.53,0.571429,0.533844,0.517857,0.053571


Weighted Accuracy: 49.43%
Weighted ROC-AUC: 0.4833
Weighted Baseline: 55.60%
Weighted Lift: -6.17%


In [23]:
# -----------------------------
# 9) 2026-ready prediction model + fixture prediction function
# -----------------------------
from collections import defaultdict, deque


if "rich_df" not in globals() or "raw_matches_df" not in globals() or "strength_map" not in globals():
    raise RuntimeError("Run Cells 15 and 18 first to build raw_matches_df, rich_df, and strength_map.")

pred_df = rich_df.copy().sort_values(["Date", "Match_ID"]).reset_index(drop=True)
pred_df["Season"] = pred_df["Date"].dt.year

# Modern-era focus usually generalizes better to upcoming IPL seasons
modern_start = 2019
pred_mod = pred_df[pred_df["Season"] >= modern_start].copy()

all_feat_candidates = [
    "A_WR_L15", "B_WR_L15", "WR_L15_Diff",
    "A_WR_EMA", "B_WR_EMA", "WR_EMA_Diff",
    "A_RR_L15", "B_RR_L15", "RR_L15_Diff",
    "A_RR_EMA", "B_RR_EMA", "RR_EMA_Diff",
    "A_Streak", "B_Streak", "Streak_Diff",
    "A_H2H_Recent",
    "A_Venue_WR", "B_Venue_WR", "Venue_WR_Diff",
    "A_Rest_Days", "B_Rest_Days", "Rest_Diff",
    "A_Toss_Form", "B_Toss_Form", "Toss_Form_Diff",
    "Elo_A", "Elo_B", "Elo_Diff", "Elo_Expected_A",
    "Team_A_Won_Toss", "Toss_Decision_Code",
    "A_BatRR_EMA", "B_BatRR_EMA", "BatRR_EMA_Diff",
    "A_BowlRRConcede_EMA", "B_BowlRRConcede_EMA", "BowlRRConcede_EMA_Diff",
    "A_BatWkRate_EMA", "B_BatWkRate_EMA", "BatWkRate_EMA_Diff",
    "A_BowlWkRate_EMA", "B_BowlWkRate_EMA", "BowlWkRate_EMA_Diff",
    "A_NetRR_EMA", "B_NetRR_EMA", "NetRR_EMA_Diff"
]
model_2026_features = [c for c in all_feat_candidates if c in pred_mod.columns]

# Validation on the latest completed season, then fit all data up to latest season
latest_season = int(pred_mod["Season"].max())
val_season = latest_season
train_2026 = pred_mod[pred_mod["Season"] < val_season].copy()
val_2026 = pred_mod[pred_mod["Season"] == val_season].copy()

if len(train_2026) < 200 or len(val_2026) < 30:
    cut = int(len(pred_mod) * 0.85)
    train_2026 = pred_mod.iloc[:cut].copy()
    val_2026 = pred_mod.iloc[cut:].copy()

min_d = train_2026["Date"].min()
age_days = (train_2026["Date"] - min_d).dt.days.clip(lower=0)
weights_2026 = np.exp(age_days / 730.0)
weights_2026 = np.asarray(weights_2026 / np.mean(weights_2026), dtype=float)

model_2026 = xgb.XGBClassifier(
    n_estimators=900,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_lambda=6,
    gamma=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model_2026.fit(train_2026[model_2026_features], train_2026["Target"], sample_weight=weights_2026)
val_proba_2026 = model_2026.predict_proba(val_2026[model_2026_features])[:, 1]

invert_2026 = False
if len(np.unique(val_2026["Target"])) > 1 and roc_auc_score(val_2026["Target"], val_proba_2026) < 0.5:
    val_proba_2026 = 1 - val_proba_2026
    invert_2026 = True

thr_grid = np.arange(0.35, 0.66, 0.01)
thr_scores = [accuracy_score(val_2026["Target"], (val_proba_2026 >= t).astype(int)) for t in thr_grid]
best_thr_2026 = float(thr_grid[int(np.argmax(thr_scores))])

val_pred_2026 = (val_proba_2026 >= best_thr_2026).astype(int)
val_acc_2026 = accuracy_score(val_2026["Target"], val_pred_2026)
val_auc_2026 = roc_auc_score(val_2026["Target"], val_proba_2026) if len(np.unique(val_2026["Target"])) > 1 else np.nan

# Refit final model on all modern data available
min_d_all = pred_mod["Date"].min()
age_days_all = (pred_mod["Date"] - min_d_all).dt.days.clip(lower=0)
weights_all = np.exp(age_days_all / 730.0)
weights_all = np.asarray(weights_all / np.mean(weights_all), dtype=float)
model_2026.fit(pred_mod[model_2026_features], pred_mod["Target"], sample_weight=weights_all)

print("\n=== 2026 PREDICTION MODEL READY ===")
print(f"Training seasons: {pred_mod['Season'].min()}-{pred_mod['Season'].max()}")
print(f"Validation rows: {len(val_2026)}")
print(f"Validation threshold: {best_thr_2026:.2f}")
print(f"Validation Accuracy: {val_acc_2026 * 100:.2f}%")
print(f"Validation ROC-AUC: {val_auc_2026:.4f}")


def _ema_from_deque(values, span, default):
    if len(values) == 0:
        return default
    return float(pd.Series(list(values), dtype="float64").ewm(span=span, adjust=False).mean().iloc[-1])


def _streak(values):
    if len(values) == 0:
        return 0
    last = values[-1]
    s = 0
    for v in reversed(values):
        if v == last:
            s += 1
        else:
            break
    return s if last == 1 else -s


def _build_state_until(cutoff_date="2025-12-31", lookback=15, h2h_lookback=20, season_regress=0.35):
    cutoff_ts = pd.Timestamp(cutoff_date)
    df_state = raw_matches_df.copy()
    df_state = df_state[df_state["Date"] <= cutoff_ts].sort_values(["Date", "Match_ID"]).reset_index(drop=True)

    team_results = defaultdict(lambda: deque(maxlen=lookback))
    team_rr = defaultdict(lambda: deque(maxlen=lookback))
    team_last_date = {}

    team_elo = defaultdict(lambda: 1500.0)
    team_last_year = {}

    h2h = defaultdict(lambda: deque(maxlen=h2h_lookback))
    venue_form = defaultdict(lambda: deque(maxlen=10))
    toss_win_form = defaultdict(lambda: deque(maxlen=lookback))
    toss_lose_form = defaultdict(lambda: deque(maxlen=lookback))

    hist_bat_rr = defaultdict(lambda: deque(maxlen=lookback))
    hist_bowl_rr = defaultdict(lambda: deque(maxlen=lookback))
    hist_bat_wk = defaultdict(lambda: deque(maxlen=lookback))
    hist_bowl_wk = defaultdict(lambda: deque(maxlen=lookback))

    for r in df_state.itertuples(index=False):
        mid = str(r.Match_ID)
        d = r.Date
        a = r.Team_A
        b = r.Team_B
        v = r.Venue
        w = r.Winner
        a_won_toss = int(r.Team_A_Won_Toss)

        year = int(d.year)
        for t in (a, b):
            if t in team_last_year and team_last_year[t] != year:
                team_elo[t] = (1 - season_regress) * team_elo[t] + season_regress * 1500.0
            team_last_year[t] = year

        y_a = 1 if w == a else 0
        exp_a = 1.0 / (1.0 + 10.0 ** ((team_elo[b] - team_elo[a]) / 400.0))
        elo_a_old, elo_b_old = team_elo[a], team_elo[b]
        team_elo[a] = elo_a_old + 24 * (y_a - exp_a)
        team_elo[b] = elo_b_old + 24 * ((1 - y_a) - (1 - exp_a))

        team_results[a].append(y_a)
        team_results[b].append(1 - y_a)

        pair = tuple(sorted([a, b]))
        h2h[pair].append(w)
        venue_form[(a, v)].append(y_a)
        venue_form[(b, v)].append(1 - y_a)

        if a_won_toss == 1:
            toss_win_form[a].append(y_a)
            toss_lose_form[b].append(1 - y_a)
        else:
            toss_lose_form[a].append(y_a)
            toss_win_form[b].append(1 - y_a)

        rr_a = rr_map2.get((mid, a), np.nan)
        rr_b = rr_map2.get((mid, b), np.nan)
        if pd.notna(rr_a):
            team_rr[a].append(float(rr_a))
        if pd.notna(rr_b):
            team_rr[b].append(float(rr_b))

        sa = strength_map.get((mid, a), None)
        sb = strength_map.get((mid, b), None)
        if sa is not None:
            hist_bat_rr[a].append(sa["bat_rr"])
            hist_bowl_rr[a].append(sa["bowl_rr_conceded"])
            hist_bat_wk[a].append(sa["bat_wk_rate"])
            hist_bowl_wk[a].append(sa["bowl_wk_rate"])
        if sb is not None:
            hist_bat_rr[b].append(sb["bat_rr"])
            hist_bowl_rr[b].append(sb["bowl_rr_conceded"])
            hist_bat_wk[b].append(sb["bat_wk_rate"])
            hist_bowl_wk[b].append(sb["bowl_wk_rate"])

        team_last_date[a] = d
        team_last_date[b] = d

    return {
        "team_results": team_results,
        "team_rr": team_rr,
        "team_last_date": team_last_date,
        "team_elo": team_elo,
        "h2h": h2h,
        "venue_form": venue_form,
        "toss_win_form": toss_win_form,
        "toss_lose_form": toss_lose_form,
        "hist_bat_rr": hist_bat_rr,
        "hist_bowl_rr": hist_bowl_rr,
        "hist_bat_wk": hist_bat_wk,
        "hist_bowl_wk": hist_bowl_wk,
    }


def _feature_row_for_fixture(team_a, team_b, venue, toss_winner=None, toss_decision="field", cutoff_date="2025-12-31"):
    st = _build_state_until(cutoff_date=cutoff_date)
    now_dt = pd.Timestamp(cutoff_date)

    team_results = st["team_results"]
    team_rr = st["team_rr"]
    team_last_date = st["team_last_date"]
    team_elo = st["team_elo"]
    h2h = st["h2h"]
    venue_form = st["venue_form"]
    toss_win_form = st["toss_win_form"]
    toss_lose_form = st["toss_lose_form"]
    hist_bat_rr = st["hist_bat_rr"]
    hist_bowl_rr = st["hist_bowl_rr"]
    hist_bat_wk = st["hist_bat_wk"]
    hist_bowl_wk = st["hist_bowl_wk"]

    a_wr_l15 = float(np.mean(team_results[team_a])) if len(team_results[team_a]) else 0.5
    b_wr_l15 = float(np.mean(team_results[team_b])) if len(team_results[team_b]) else 0.5
    a_wr_ema = _ema_from_deque(team_results[team_a], 8, 0.5)
    b_wr_ema = _ema_from_deque(team_results[team_b], 8, 0.5)

    a_rr_l15 = float(np.mean(team_rr[team_a])) if len(team_rr[team_a]) else 8.0
    b_rr_l15 = float(np.mean(team_rr[team_b])) if len(team_rr[team_b]) else 8.0
    a_rr_ema = _ema_from_deque(team_rr[team_a], 8, 8.0)
    b_rr_ema = _ema_from_deque(team_rr[team_b], 8, 8.0)

    pair = tuple(sorted([team_a, team_b]))
    if len(h2h[pair]) == 0:
        a_h2h_recent = 0.5
    else:
        a_h2h_recent = sum(1 for w in h2h[pair] if w == team_a) / len(h2h[pair])

    a_venue_wr = float(np.mean(venue_form[(team_a, venue)])) if len(venue_form[(team_a, venue)]) else 0.5
    b_venue_wr = float(np.mean(venue_form[(team_b, venue)])) if len(venue_form[(team_b, venue)]) else 0.5

    a_rest_days = int((now_dt - team_last_date[team_a]).days) if team_a in team_last_date else 7
    b_rest_days = int((now_dt - team_last_date[team_b]).days) if team_b in team_last_date else 7

    if toss_winner is None:
        team_a_won_toss = 0
    else:
        team_a_won_toss = int(toss_winner == team_a)

    if team_a_won_toss == 1:
        a_toss_form = float(np.mean(toss_win_form[team_a])) if len(toss_win_form[team_a]) else 0.5
        b_toss_form = float(np.mean(toss_lose_form[team_b])) if len(toss_lose_form[team_b]) else 0.5
    else:
        a_toss_form = float(np.mean(toss_lose_form[team_a])) if len(toss_lose_form[team_a]) else 0.5
        b_toss_form = float(np.mean(toss_win_form[team_b])) if len(toss_win_form[team_b]) else 0.5

    elo_a = float(team_elo[team_a])
    elo_b = float(team_elo[team_b])
    elo_expected_a = 1.0 / (1.0 + 10.0 ** ((elo_b - elo_a) / 400.0))

    a_bat_rr_ema = _ema_from_deque(hist_bat_rr[team_a], 8, 8.0)
    b_bat_rr_ema = _ema_from_deque(hist_bat_rr[team_b], 8, 8.0)
    a_bowl_rr_ema = _ema_from_deque(hist_bowl_rr[team_a], 8, 8.0)
    b_bowl_rr_ema = _ema_from_deque(hist_bowl_rr[team_b], 8, 8.0)
    a_bat_wk_ema = _ema_from_deque(hist_bat_wk[team_a], 8, 0.04)
    b_bat_wk_ema = _ema_from_deque(hist_bat_wk[team_b], 8, 0.04)
    a_bowl_wk_ema = _ema_from_deque(hist_bowl_wk[team_a], 8, 0.04)
    b_bowl_wk_ema = _ema_from_deque(hist_bowl_wk[team_b], 8, 0.04)

    row = {
        "A_WR_L15": a_wr_l15,
        "B_WR_L15": b_wr_l15,
        "WR_L15_Diff": a_wr_l15 - b_wr_l15,
        "A_WR_EMA": a_wr_ema,
        "B_WR_EMA": b_wr_ema,
        "WR_EMA_Diff": a_wr_ema - b_wr_ema,
        "A_RR_L15": a_rr_l15,
        "B_RR_L15": b_rr_l15,
        "RR_L15_Diff": a_rr_l15 - b_rr_l15,
        "A_RR_EMA": a_rr_ema,
        "B_RR_EMA": b_rr_ema,
        "RR_EMA_Diff": a_rr_ema - b_rr_ema,
        "A_Streak": _streak(team_results[team_a]),
        "B_Streak": _streak(team_results[team_b]),
        "Streak_Diff": _streak(team_results[team_a]) - _streak(team_results[team_b]),
        "A_H2H_Recent": a_h2h_recent,
        "A_Venue_WR": a_venue_wr,
        "B_Venue_WR": b_venue_wr,
        "Venue_WR_Diff": a_venue_wr - b_venue_wr,
        "A_Rest_Days": a_rest_days,
        "B_Rest_Days": b_rest_days,
        "Rest_Diff": a_rest_days - b_rest_days,
        "A_Toss_Form": a_toss_form,
        "B_Toss_Form": b_toss_form,
        "Toss_Form_Diff": a_toss_form - b_toss_form,
        "Elo_A": elo_a,
        "Elo_B": elo_b,
        "Elo_Diff": elo_a - elo_b,
        "Elo_Expected_A": elo_expected_a,
        "Team_A_Won_Toss": team_a_won_toss,
        "Toss_Decision_Code": 1 if str(toss_decision).lower() == "bat" else 0,
        "A_BatRR_EMA": a_bat_rr_ema,
        "B_BatRR_EMA": b_bat_rr_ema,
        "BatRR_EMA_Diff": a_bat_rr_ema - b_bat_rr_ema,
        "A_BowlRRConcede_EMA": a_bowl_rr_ema,
        "B_BowlRRConcede_EMA": b_bowl_rr_ema,
        "BowlRRConcede_EMA_Diff": a_bowl_rr_ema - b_bowl_rr_ema,
        "A_BatWkRate_EMA": a_bat_wk_ema,
        "B_BatWkRate_EMA": b_bat_wk_ema,
        "BatWkRate_EMA_Diff": a_bat_wk_ema - b_bat_wk_ema,
        "A_BowlWkRate_EMA": a_bowl_wk_ema,
        "B_BowlWkRate_EMA": b_bowl_wk_ema,
        "BowlWkRate_EMA_Diff": a_bowl_wk_ema - b_bowl_wk_ema,
        "A_NetRR_EMA": a_bat_rr_ema - a_bowl_rr_ema,
        "B_NetRR_EMA": b_bat_rr_ema - b_bowl_rr_ema,
        "NetRR_EMA_Diff": (a_bat_rr_ema - a_bowl_rr_ema) - (b_bat_rr_ema - b_bowl_rr_ema),
    }

    return pd.DataFrame([row])


def predict_ipl_2026(team_a, team_b, venue, toss_winner=None, toss_decision="field", cutoff_date="2025-12-31"):
    x_row = _feature_row_for_fixture(
        team_a=team_a,
        team_b=team_b,
        venue=venue,
        toss_winner=toss_winner,
        toss_decision=toss_decision,
        cutoff_date=cutoff_date,
    )

    x_row = x_row.reindex(columns=model_2026_features, fill_value=0)
    p_a = float(model_2026.predict_proba(x_row)[:, 1][0])
    if invert_2026:
        p_a = 1.0 - p_a
    p_b = 1.0 - p_a

    pred_label = team_a if p_a >= best_thr_2026 else team_b
    return {
        "team_a": team_a,
        "team_b": team_b,
        "venue": venue,
        "toss_winner": toss_winner,
        "toss_decision": toss_decision,
        "prob_team_a_win": round(p_a, 4),
        "prob_team_b_win": round(p_b, 4),
        "predicted_winner": pred_label,
        "decision_threshold": round(best_thr_2026, 2),
    }


# Example usage (edit teams/venue to your fixture)
example_pred = predict_ipl_2026(
    team_a="Mumbai Indians",
    team_b="Chennai Super Kings",
    venue="Wankhede Stadium",
    toss_winner=None,
    toss_decision="field",
    cutoff_date="2025-12-31",
)
print("\nExample 2026 fixture prediction:")
print(example_pred)


=== 2026 PREDICTION MODEL READY ===
Training seasons: 2019-2025
Validation rows: 70
Validation threshold: 0.64
Validation Accuracy: 55.71%
Validation ROC-AUC: 0.5627

Example 2026 fixture prediction:
{'team_a': 'Mumbai Indians', 'team_b': 'Chennai Super Kings', 'venue': 'Wankhede Stadium', 'toss_winner': None, 'toss_decision': 'field', 'prob_team_a_win': 0.5997, 'prob_team_b_win': 0.4003, 'predicted_winner': 'Chennai Super Kings', 'decision_threshold': 0.64}


In [26]:
# -----------------------------
# 10) Batch predict full IPL 2026 fixtures table
# -----------------------------
from pathlib import Path

fixtures_path = Path("fixtures_2026.csv")
output_path = Path("ipl_2026_predictions.csv")

required_cols = ["Team_A", "Team_B", "Venue"]
optional_cols = ["Toss_Winner", "Toss_Decision", "Cutoff_Date", "Match_Date"]

if not fixtures_path.exists():
    template = pd.DataFrame([
        {
            "Team_A": "Mumbai Indians",
            "Team_B": "Chennai Super Kings",
            "Venue": "Wankhede Stadium",
            "Toss_Winner": "",
            "Toss_Decision": "field",
            "Cutoff_Date": "2025-12-31",
            "Match_Date": "2026-03-22"
        },
        {
            "Team_A": "Royal Challengers Bengaluru",
            "Team_B": "Kolkata Knight Riders",
            "Venue": "M. Chinnaswamy Stadium",
            "Toss_Winner": "",
            "Toss_Decision": "field",
            "Cutoff_Date": "2025-12-31",
            "Match_Date": "2026-03-23"
        }
    ])
    template.to_csv(fixtures_path, index=False)
    print("Created template fixtures_2026.csv. Fill it with your real 2026 fixtures, then rerun this cell.")
else:
    fx = pd.read_csv(fixtures_path)

    missing = [c for c in required_cols if c not in fx.columns]
    if missing:
        raise ValueError(f"fixtures_2026.csv is missing required columns: {missing}")

    for c in optional_cols:
        if c not in fx.columns:
            fx[c] = ""

    rows = []
    for r in fx.itertuples(index=False):
        team_a = str(r.Team_A).strip()
        team_b = str(r.Team_B).strip()
        venue = str(r.Venue).strip()

        toss_winner = str(r.Toss_Winner).strip() if pd.notna(r.Toss_Winner) else ""
        toss_winner = toss_winner if toss_winner != "" else None

        toss_decision = str(r.Toss_Decision).strip().lower() if pd.notna(r.Toss_Decision) and str(r.Toss_Decision).strip() else "field"

        cutoff_date = str(r.Cutoff_Date).strip() if pd.notna(r.Cutoff_Date) and str(r.Cutoff_Date).strip() else "2025-12-31"
        match_date = str(r.Match_Date).strip() if pd.notna(r.Match_Date) else ""

        pred = predict_ipl_2026(
            team_a=team_a,
            team_b=team_b,
            venue=venue,
            toss_winner=toss_winner,
            toss_decision=toss_decision,
            cutoff_date=cutoff_date,
        )

        rows.append({
            "Match_Date": match_date,
            "Team_A": pred["team_a"],
            "Team_B": pred["team_b"],
            "Venue": pred["venue"],
            "Toss_Winner": pred["toss_winner"],
            "Toss_Decision": pred["toss_decision"],
            "Prob_Team_A_Win": pred["prob_team_a_win"],
            "Prob_Team_B_Win": pred["prob_team_b_win"],
            "Predicted_Winner": pred["predicted_winner"],
            "Decision_Threshold": pred["decision_threshold"]
        })

    pred_df_2026 = pd.DataFrame(rows)
    pred_df_2026 = pred_df_2026.sort_values("Prob_Team_A_Win", ascending=False).reset_index(drop=True)
    pred_df_2026.to_csv(output_path, index=False)

    print(f"Scored {len(pred_df_2026)} fixtures.")
    print(f"Saved predictions to {output_path}")
    display(pred_df_2026.head(20))

Scored 20 fixtures.
Saved predictions to ipl_2026_predictions.csv


,Match_Date,Team_A,Team_B,Venue,Toss_Winner,Toss_Decision,Prob_Team_A_Win,Prob_Team_B_Win,Predicted_Winner,Decision_Threshold
0,2026-03-31,Punjab Kings,Gujarat Titans,New Chandigarh,None,field,0.7757,0.2243,Punjab Kings,0.64
1,2026-04-05,Royal Challengers Bengaluru,Chennai Super Kings,Bengaluru,None,field,0.6838,0.3162,Royal Challengers Bengaluru,0.64
2,2026-04-08,Delhi Capitals,Gujarat Titans,Delhi,None,field,0.6698,0.3302,Delhi Capitals,0.64
3,2026-04-10,Rajasthan Royals,Royal Challengers Bengaluru,Guwahati,None,field,0.6563,0.3437,Rajasthan Royals,0.64
4,2026-03-29,Mumbai Indians,Kolkata Knight Riders,Mumbai,None,field,0.6308,0.3692,Kolkata Knight Riders,0.64
5,2026-04-01,Lucknow Super Giants,Delhi Capitals,Lucknow,None,field,0.6064,0.3936,Delhi Capitals,0.64
6,2026-04-12,Lucknow Super Giants,Gujarat Titans,Lucknow,None,field,0.4697,0.5303,Gujarat Titans,0.64
7,2026-04-12,Mumbai Indians,Royal Challengers Bengaluru,Mumbai,None,field,0.4604,0.5396,Royal Challengers Bengaluru,0.64
8,2026-04-11,Chennai Super Kings,Delhi Capitals,Chennai,None,field,0.4387,0.5613,Delhi Capitals,0.64
9,2026-04-09,Kolkata Knight Riders,Lucknow Super Giants,Kolkata,None,field,0.4308,0.5692,Lucknow Super Giants,0.64


In [27]:
import pandas as pd
import json

# 1. Read the predictions file Copilot just made
predictions_df = pd.read_csv('ipl_2026_predictions.csv')

react_data = []

# 2. Loop through the predictions and format them for React
for index, row in predictions_df.iterrows():
    # Format the ID (e.g., IPL2026-001)
    match_id = f"IPL2026-{str(index + 1).zfill(3)}"
    
    # Extract the predicted winner and the probability/confidence
    # (Adjust 'Predicted_Winner' and 'Win_Probability' if Copilot named them differently in the CSV)
    predicted_winner = row.get('Predicted_Winner', 'To Be Updated')
    
    # Handle confidence formatting (convert 0.58 to 58)
    confidence_raw = row.get('Win_Probability', 50)
    if isinstance(confidence_raw, float) and confidence_raw < 1:
        confidence = int(confidence_raw * 100)
    else:
        confidence = int(confidence_raw)

    # Determine Toss Impact dynamically
    toss_impact = "High" if confidence > 55 and pd.isna(row.get('Toss_Winner')) else "TBD"
    
    # Format the final object for the frontend
    match_obj = {
        "id": match_id,
        "teamA": row['Team_A'],
        "teamB": row['Team_B'],
        "venue": row['Venue'],
        "date": str(row.get('Match_Date', 'TBD')),
        "kickoff": "7:30 PM", # Defaulting to standard IPL time
        "predictedWinner": predicted_winner,
        "confidence": confidence,
        "tossImpact": "Post-Toss" if pd.notna(row.get('Toss_Winner')) else toss_impact,
        "keyDrivers": [
            f"Pre-match model confidence: {confidence}%",
            "Elo & Form EMA factored"
        ]
    }
    react_data.append(match_obj)

# 3. Output the exact JavaScript code you need to paste into your React app
print("✅ Copy and paste this array into your React code (replace the UPCOMING_PREDICTIONS array):\n")
print("const UPCOMING_PREDICTIONS = " + json.dumps(react_data, indent=2) + ";")

✅ Copy and paste this array into your React code (replace the UPCOMING_PREDICTIONS array):

const UPCOMING_PREDICTIONS = [
  {
    "id": "IPL2026-001",
    "teamA": "Punjab Kings",
    "teamB": "Gujarat Titans",
    "venue": "New Chandigarh",
    "date": "2026-03-31",
    "kickoff": "7:30 PM",
    "predictedWinner": "Punjab Kings",
    "confidence": 50,
    "tossImpact": "TBD",
    "keyDrivers": [
      "Pre-match model confidence: 50%",
      "Elo & Form EMA factored"
    ]
  },
  {
    "id": "IPL2026-002",
    "teamA": "Royal Challengers Bengaluru",
    "teamB": "Chennai Super Kings",
    "venue": "Bengaluru",
    "date": "2026-04-05",
    "kickoff": "7:30 PM",
    "predictedWinner": "Royal Challengers Bengaluru",
    "confidence": 50,
    "tossImpact": "TBD",
    "keyDrivers": [
      "Pre-match model confidence: 50%",
      "Elo & Form EMA factored"
    ]
  },
  {
    "id": "IPL2026-003",
    "teamA": "Delhi Capitals",
    "teamB": "Gujarat Titans",
    "venue": "Delhi",
    "date"